# Testing a vLLM Model Endpoint on Klique

This notebook sends test requests to a model served by the **vLLM Model Deployment** app and shows how to read the response, including the `<think>` block that reasoning models such as Qwen3 return.

**What you need**
1. A running vLLM Model Deployment instance with an **Active** endpoint. See [vLLM Model Deployment](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment) and [Model Endpoints](https://klique.ai/docs/latest/docs/webapp/webapp_model_endpoints).
2. The endpoint URL, from the app's *Deployed models* table or the Model Endpoints table.
3. An **AI Application Gateway token**. Generate it in Settings, in the *AI APPLICATION GATEWAY* section ([steps](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_profile#ai-application-gateway-tokens)): click **Generate a Token**, set a `Label` and an `Expiration` in days, then click `Generate`. The token is copied to your clipboard.
4. A machine that can reach the endpoint's network. The host name ends in `.internal.allegro.ai`.

> **Keep the token out of the notebook.** Section 2 reads it from the `.env` file at the project root, which you created in `01_Deploy_a_Model_with_vLLM.ipynb`. Do not paste a token into a cell: it is saved in the `.ipynb` file and in any copy you share. To revoke a token, hover over its row in the Settings token table and click the trash icon.

**Documented vs. not documented.** The Klique docs show only a `/v1/completions` curl example for this app. The `/v1/chat/completions` calls below follow vLLM's standard OpenAI-compatible API; the Klique docs do not state that route is exposed, so a `404` there means it is not.

## 1. Install

The notebook needs only `requests`.

In [ ]:
%pip install -q requests

## 2. Configure

Load the endpoint URL, the model name, and the token from the `.env` file at the project root, through `config.py`. If you have not created `.env` yet, do Section 1.6 of `01_Deploy_a_Model_with_vLLM.ipynb` first.

- `BASE_URL` is the endpoint URL ending in `/v1`.
- `MODEL` must match the **Model Endpoint Name** you set when you launched the app.
- The token is sent as a bearer token in the `Authorization` header.

In [ ]:
import re
import sys

import requests

sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as BASE_URL, KLIQUE_MODEL as MODEL, KLIQUE_GATEWAY_TOKEN as TOKEN

TIMEOUT = 60  # seconds per request

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

print("BASE_URL:", BASE_URL)
print("MODEL:   ", MODEL)
print(f"TOKEN:    {TOKEN[:8]}...")

## 3. Helper functions

- `post()` sends a request and calls `raise_for_status()`, which raises an error for any HTTP `4xx` or `5xx` response. Without it, a failed call would surface later as a confusing missing-key error.
- `split_think()` separates the `<think>...</think>` block from the final answer. It also handles a reply that was cut off before `</think>`; in that case the answer is empty, so raise `max_tokens`.

In [ ]:
def post(path, payload):
    """POST a JSON payload to BASE_URL + path and return the parsed JSON reply."""
    r = requests.post(f"{BASE_URL}{path}", headers=HEADERS, json=payload, timeout=TIMEOUT)
    if not r.ok:
        # Show the server's explanation, e.g. 401 (bad or expired token) or 404 (wrong path).
        print(f"HTTP {r.status_code}: {r.text[:500]}")
    r.raise_for_status()
    return r.json()


def split_think(text):
    """Return (thinking, answer) from a reply that may contain a <think> block."""
    m = re.search(r"<think>(.*?)(?:</think>|$)", text, flags=re.DOTALL)
    thinking = m.group(1).strip() if m else ""
    answer = re.sub(r"<think>.*?(?:</think>|$)", "", text, flags=re.DOTALL).strip()
    return thinking, answer


def print_usage(data):
    """Print token counts, if the server returned them."""
    usage = data.get("usage", {})
    print("-" * 40)
    print("Metrics")
    print("-" * 40)
    print(f"Prompt tokens:     {usage.get('prompt_tokens')}")
    print(f"Completion tokens: {usage.get('completion_tokens')}")
    print(f"Total tokens:      {usage.get('total_tokens')}")

## 4. Completions: send a raw prompt

This matches the curl example on the app's dashboard (`POST /v1/completions`). The `prompt` is a single string with no roles, so there is no place for a system prompt here.

`max_tokens` is the most tokens the model may generate. It is optional, and the Klique docs do not say what limit applies when it is left out, so set it explicitly. Qwen3 can spend many tokens on reasoning, so a low value can leave you with a truncated answer.

In [ ]:
data = post("/completions", {
    "model": MODEL,
    "prompt": "Hello world!",
    "max_tokens": 256,
})
print(data["choices"][0]["text"])

## 5. Chat completions: add a system prompt

A system prompt is a message with `"role": "system"`, and it works only on the chat endpoint. The Klique docs do not describe a system prompt setting; this is vLLM's standard behavior.

Reasoning models put their thinking in the reply. The next cell shows the raw reply, the thinking, and the answer separately.

In [ ]:
data = post("/chat/completions", {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": "You are a concise assistant. Answer in a limerick."},
        {"role": "user", "content": "What does the fox say?"},
    ],
    "max_tokens": 512,
})

raw = data["choices"][0]["message"]["content"]
thinking, answer = split_think(raw)

print("ANSWER")
print(answer)
print()
print(f"(thinking: {len(thinking)} characters, hidden. Print `thinking` to read it.)")
print()
print_usage(data)

## 6. Turn the `<think>` output off or handle it differently

There are three ways to deal with reasoning output. Only the first is documented by Klique.

1. **Parse it on the server (documented).** The [vLLM Model Deployment page](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment) has the fields *Enable Reasoning* ("Enables model to generate reasoning content") and *Reasoning Parser* ("Select the reasoning parser depending on the model that you're using... Required if `Reasoning` is enabled"). The page does not say which parser value fits Qwen3. You must redeploy to change these.
2. **Ask the model not to think (not in the Klique docs).** For Qwen3 on vLLM's chat endpoint, pass `chat_template_kwargs`. The next cell tries it. If the reply still has a `<think>` block, the option is not supported in your setup.
3. **Strip it in code.** Use `split_think()` from section 3, as in section 5.

In [ ]:
data = post("/chat/completions", {
    "model": MODEL,
    "messages": [{"role": "user", "content": "What does the fox say?"}],
    "max_tokens": 256,
    "chat_template_kwargs": {"enable_thinking": False},
})

raw = data["choices"][0]["message"]["content"]
print(raw)
print()
print("Still contains <think>:", "<think>" in raw)

## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `401` | The token is wrong, expired, or revoked. Generate a new one in Settings. |
| `404` | The URL or path is wrong, or the route does not expose that endpoint (for example `/chat/completions`). Check `BASE_URL` against the Deployed models table. |
| Connection error or timeout | The machine cannot reach the `.internal.allegro.ai` network, the app is still loading, or the idle shutdown already stopped it. Check the status in [Model Endpoints](https://klique.ai/docs/latest/docs/webapp/webapp_model_endpoints). |
| Empty answer after `split_think()` | The reply was cut off inside `<think>`. Raise `max_tokens`. |
| The model name is rejected | `MODEL` does not match the **Model Endpoint Name** from the launch form. |

## Related docs
- [vLLM Model Deployment](https://klique.ai/docs/latest/docs/webapp/applications/apps_model_deployment)
- [Model Endpoints](https://klique.ai/docs/latest/docs/webapp/webapp_model_endpoints)
- [AI Application Gateway](https://klique.ai/docs/latest/docs/whats_klique/appgw)
- [Settings: AI Application Gateway Tokens](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_profile#ai-application-gateway-tokens)